In [2]:
import sys
sys.path.append("../src")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from features import make_features, FEATURES
from evaluation import time_split

PRICE_COLS = ["Open", "High", "Low", "Close", "Volume"]
df = pd.read_csv("../data/LUCK_target.csv", index_col="Date", parse_dates=True)

feats = make_features(df[PRICE_COLS])
print(feats.isna().sum())
print(feats.describe().round(4))


ret_1        1
ret_5        5
vol_10      10
vol_chg     19
hl_range     0
ma_dist     19
dtype: int64
           ret_1      ret_5     vol_10    vol_chg   hl_range    ma_dist
count  2883.0000  2879.0000  2874.0000  2865.0000  2884.0000  2865.0000
mean      0.0007     0.0037     0.0185     0.0215     0.0279     0.0055
std       0.0207     0.0491     0.0088     0.7993     0.0152     0.0544
min      -0.0994    -0.2815     0.0026    -0.9444     0.0000    -0.3304
25%      -0.0104    -0.0241     0.0122    -0.4933     0.0171    -0.0280
50%      -0.0005     0.0007     0.0171    -0.2007     0.0242     0.0029
75%       0.0104     0.0281     0.0228     0.2979     0.0354     0.0380
max       0.1000     0.2625     0.0622     7.6154     0.1513     0.2227


In [3]:
f_full = make_features(df[PRICE_COLS])
f_cut = make_features(df[PRICE_COLS].iloc[:2000])
pd.testing.assert_frame_equal(f_full.iloc[:2000], f_cut)
print("PASS: no feature uses future rows")



PASS: no feature uses future rows


### Look-Ahead Bias Check

This test checks whether `make_features()` accidentally uses future data when calculating historical features.

- `f_full`: Generates features using the entire dataset.
- `f_cut`: Generates features using only the first 2,000 rows.
- `assert_frame_equal()`: Checks whether the first 2,000 feature rows are identical in both DataFrames.

**Why?** If a feature accidentally uses future observations, its calculated values may differ when those future rows are unavailable.

**PASS** means the feature calculations are independent of future rows. This checks feature-level look-ahead bias, not leakage across the entire ML pipeline.

In [4]:
data = feats.join(df[["target", "next_return"]])
dev, _ = time_split(data)
dev = dev.dropna()

cv = TimeSeriesSplit(n_splits=5, gap=1)
honest = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

acc_honest = cross_val_score(honest, dev[FEATURES], dev["target"], cv=cv, scoring="accuracy").mean()
acc_cheat = cross_val_score(honest, dev[FEATURES + ["next_return"]], dev["target"], cv=cv, scoring="accuracy").mean()

print("honest features accuracy :", round(acc_honest, 4))
print("with next_return (cheat) :", round(acc_cheat, 4))


honest features accuracy : 0.5373
with next_return (cheat) : 0.9843


### Understanding the Data Leakage Experiment and Validation

This experiment compares two logistic regression models:

- **Honest model:** Uses six features calculated from information available on the prediction day.
- **Cheat model:** Uses the same six features plus `next_return`, which contains the actual next-day return.

Since `target` is derived from whether `next_return` is positive, including `next_return` gives the model information about the answer. This is an example of **data leakage**.

The honest model achieved 54.01% average validation accuracy, while the cheat model achieved 98.43%. The cheat model's high accuracy is misleading because it uses future information.

**Important:** These results are preliminary. We must use `gap=1` in `TimeSeriesSplit` because each target depends on the following trading day's closing price. This prevents the final training observation in each fold from using an outcome from the validation period.

Cross-validation trains temporary model copies to evaluate performance. We will train our final model separately after selecting its configuration.